# StormVerify-Py: NWS Warning Verification & Spatial Audit

**Event Date:** April 27, 2026  
**Warning Types:** Tornado Warnings (TOR) and Severe Thunderstorm Warnings (SVR)  
**Data Source:** Iowa Environmental Mesonet (IEM)

This notebook walks through a spatial pipeline that evaluates whether NWS convective warnings were verified by ground-truth Local Storm Reports (LSRs). A warning is verified only if a qualifying report occurs **inside the polygon** while the warning is **still active**.

---

### Verification Rules

| Warning Type | Verified By |
|---|---|
| Tornado Warning (TOR) | Tornado reports only |
| Severe Thunderstorm Warning (SVR) | Hail OR damaging wind reports |

> **Note:** Storm surveys for April 27 are ongoing. Tornado report counts may increase as surveys conclude.

## Step 1: Imports and Environment Setup

Import your libraries and set up the directory structure for raw and processed data. You'll need tools for HTTP requests, spatial data I/O, DuckDB, and visualization. Make sure your `data/raw` and `data/processed` folders exist before moving on.

In [1]:
import requests
import duckdb
import pandas as pd
import geopandas as gpd
import os
import folium
from shapely import wkb

raw_folder = os.path.join("data", "raw")
processed_folder = os.path.join("data", "processed")
figures_folder = os.path.join("figures")

## Step 2: Ingest Data from IEM

Pull two datasets from the IEM API for the April 27, 2026 24-hour window:

- **LSRs** — Local Storm Reports (point data, ground-truth observations)
- **Storm-Based Warnings** — polygon warnings issued by NWS WFOs

Use the **Storm Based Warnings** option in the IEM warning archive — this gives you actual drawn polygons rather than county outlines, which matters a lot for spatial accuracy.

Save both files as GeoJSON to your `data/raw/` directory.

For constructing parameters for IEM requests, see URLs below:

LSRs: https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py?help

Warnings: https://mesonet.agron.iastate.edu/cgi-bin/request/gis/watchwarn.py?help

In [2]:
# Define parameters for appending to the URL. Documentation can be found on IEM site (See links above)
param_lsr = {'magge':'','wfo':'ALL','state':'_ALL','year1':2026, 'month1':4,'day1':27,'hour1':12,'minute1':00,'year2':2026, 'month2':4,'day2':28,'hour2':11,'minute2':59, 'type':['TORNADO','HAIL','TSTM WND DMG','TSTM WND GST'] , 'fmt':'csv'}
param_warn = {'accept':'shapefile', 'wfo':'ALL', 'limit1':True,'year1':2026, 'month1':4,'day1':27,'hour1':12,'minute1':00,'year2':2026, 'month2':4,'day2':28,'hour2':11,'minute2':59}

# Grab LSR CSV
with requests.get('https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py', params=param_lsr, stream=True) as response:
    response.raise_for_status()

    lsr_filename = os.path.join('data','raw','lsr_iem.csv') 

    with open(lsr_filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("LSR CSV Downloaded!")

# Grab warning zipped shapefile
with requests.get('https://mesonet.agron.iastate.edu/cgi-bin/request/gis/watchwarn.py', params=param_warn, stream=True) as response:
    response.raise_for_status()

    warnings_filename = os.path.join('data', 'raw', 'warnings.zip')

    with open(warnings_filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("Storm Warning Shapefile Downloaded!")


LSR CSV Downloaded!
Storm Warning Shapefile Downloaded!


## Step 3: Preprocess and Filter

Load the raw GeoJSON files and filter both datasets down to what you actually need.

**Warnings:** keep only TOR and SVR. Drop everything else.

**LSRs:** keep only report types that can verify one of your two warning types — tornado, hail, large hail, wind, and high wind reports.

This is also a good moment to inspect the data. Check your CRS on both layers, look at your timestamp columns, and make sure geometries are present and reasonable. Catching issues here saves a lot of confusion later.

In [3]:
# Load into geopandas
gdf_lsr = gpd.read_file(lsr_filename) # LSR CSV
gdf_warn = gpd.read_file(r'zip://data/raw/warnings.zip') # Storm Warnings shapefile

# Field type conversions
gdf_lsr['VALID'] = pd.to_datetime(gdf_lsr['VALID']) # convert string date/time field (VALID) to datetime
gdf_warn['ISSUED'] = pd.to_datetime(gdf_warn['ISSUED']) # convert string date/time field (ISSUED ) to datetime to match VALID
gdf_warn['EXPIRED'] = pd.to_datetime(gdf_warn['EXPIRED'])# convert string date/time field (EXPIRED) to datetime to match VALID
gdf_warn['INIT_ISS'] = pd.to_datetime(gdf_warn['INIT_ISS'])
gdf_warn['INIT_EXP'] = pd.to_datetime(gdf_warn['INIT_EXP'])
gdf_lsr['LAT'] = pd.to_numeric(gdf_lsr['LAT'],downcast='float') # Convert lat/lon fields from string to float
gdf_lsr['LON'] = pd.to_numeric(gdf_lsr['LON'],downcast='float')

# 'Dropping' LSR event with erroneous LAT value of 8.9 (being in Jackson, MO, which doesn't make sense geographically)
gdf_lsr = gdf_lsr[gdf_lsr['LAT'] > 9]

# 'Dropping' warnings except TOR and SVR
gdf_warn = gdf_warn.query("PHENOM in ['TO', 'SV']")

# Build point geometries in LSR gdf
gdf_lsr = gpd.GeoDataFrame(gdf_lsr,geometry=gpd.points_from_xy(gdf_lsr.LON, gdf_lsr.LAT),crs="EPSG:4326")

In [4]:
print(gdf_warn['PHENOM'].unique())

['TO' 'SV']


## Step 4: Convert to GeoParquet

DuckDB can read GeoJSON directly, but GeoParquet is much faster for spatial joins. Write both filtered datasets to `data/processed/` as Parquet files.

GeoPandas handles this natively with `.to_parquet()` — just make sure `pyarrow` is installed.

In [5]:
lsr_parquet = os.path.join("data", "processed", "lsr.parquet")
warn_parquet = os.path.join("data","processed", "warn.parquet")

# Convert geodataframes to parquet files
gdf_lsr.to_parquet(lsr_parquet)
gdf_warn.to_parquet(warn_parquet)

## Step 5: Initialize DuckDB and Load Spatial Extension

Open a DuckDB connection and install and load the spatial extension. DuckDB reads Parquet files directly via `read_parquet()` in SQL — no traditional table import needed.

Run a quick row count on both files to confirm they loaded as expected.

In [6]:
#Open DuckDB connection
con = duckdb.connect()

# Install and Load Spatial Connection
con.install_extension("spatial")
con.load_extension("spatial")

# Load data into in-memory duckdb tables
ddb_lsr = con.read_parquet(lsr_parquet)
ddb_warn = con.read_parquet(warn_parquet)

# Quick rowcounts
print(con.sql('''SELECT COUNT(*) FROM ddb_lsr'''))
print(con.sql('''SELECT COUNT(*) FROM ddb_warn'''))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          739 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          421 │
└──────────────┘



In [7]:
# First 5 records from each table
print(con.sql('''SELECT * FROM ddb_lsr LIMIT 5'''))
print(con.sql('''SELECT * FROM ddb_warn LIMIT 5'''))

┌─────────────────────┬──────────────────┬───────┬────────┬─────────┬─────────┬──────────┬──────────────┬──────────────────┬──────────┬─────────┬─────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬──────────┬───────────┬───────────────────────────────────────────────┬───────────────────┐
│        VALID        │      VALID2      │  LAT  │  LON   │   MAG   │   WFO   │ TYPECODE │   TYPETEXT   │       CITY       │  COUNTY  │  STATE  │     SOURCE      │                                                                     REMARK                                                                      │   UGC   │ UGCNAME  │ QUALIFIER │                   geometry                    │ __index_level_0__ │
│    timestamp_ns     │     varchar      │ float │ float  │ varchar │ varchar │ varchar  │   varchar    │     varchar      │ varchar  │ varchar │     varchar     │               

## Step 6: Spatial Join with Temporal Constraint

This is the core of the pipeline. Join LSR points to warning polygons using two conditions at once:

1. **Spatial:** the report point must fall inside the warning polygon
2. **Temporal:** the report time must fall between the warning's issue time and expire time

Use a `LEFT JOIN` so warnings with zero matching reports still appear in results — those are your false alarms.

The output at this stage is one row per warning-report match. A warning with three reports inside it appears three times. That's expected — you'll collapse it in the next step.

A single report can match more than one warning if it falls inside overlapping polygons that are both active at the same time. This is intentional and reflects real NWS operational practice.

In [8]:
# Perform Spatio-Temporal Join
lsr_warn_join = con.sql('''
    SELECT
        w.wfo,
        w.etn,
        w.issued AS WARN_ISSUED,
        w.expired AS WARN_EXPIRED,
        l.valid as LSR_TIME,
        l.typetext,
        w.phenom AS WARN_TYPE,
        w.geometry AS warn_geom,
        l.geometry AS lsr_geom
FROM ddb_warn w
LEFT JOIN ddb_lsr l 
        ON ST_Intersects(w.geometry, l.geometry) -- Spatial Condition ()
        AND l.valid >= w.issued  -- Temporal condition start
        AND l.valid <= w.expired  -- Temporal condition end
         ''')

lsr_warn_join.show()

┌─────────┬───────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────┬───────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────┐
│   WFO   │  ETN  │     WARN_ISSUED     │    WARN_EXPIRED     │      LSR_TIME       │   TYPETEXT   │ WARN_TYPE │                                                                                                                                                   warn_geom                                                                                                                                                   │                   lsr_geom                    │
│ varchar │ int32 │    timestamp_ns     │    timestamp_ns     │    tim

## Step 7: Aggregate Report Counts by Warning

Collapse the join output to one row per warning. Rather than a single total count, produce **separate counts by report type** — tornado reports, hail reports, and wind reports each get their own column.

This is what allows the classification step to distinguish between a Tornado Warning that had hail inside it versus one that had an actual tornado report. A single total count would lose that distinction.

Build this as a CTE that wraps your join from Step 6.

In [9]:
# Aggregate Report Counts
warn_lsr_totals= con.sql('''SELECT wfo, etn, warn_type, warn_geom,
                        COUNT_IF(TYPETEXT = 'TORNADO') AS tornado_reports,
                        COUNT_IF(TYPETEXT = 'HAIL') AS hail_reports,
                        COUNT_IF(TYPETEXT = 'TSTM WND DMG' or TYPETEXT = 'TSTM WND GST') as wind_reports,
                        COUNT(TYPETEXT) as total_reports
                    FROM lsr_warn_join
                    GROUP BY wfo, etn, warn_type, warn_geom
                    ORDER BY total_reports DESC                     
                     ''')

warn_lsr_totals.show()

┌─────────┬───────┬───────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────┬──────────────┬──────────────┬───────────────┐
│   WFO   │  ETN  │ WARN_TYPE │                                                                                                                                                  warn_geom                                                                                                                                                  │ tornado_reports │ hail_reports │ wind_reports │ total_reports │
│ varchar │ int32 │  varchar  │                                                                                                                                                  geometry                   

## Step 8: Classify Warnings

With per-type report counts on each row, apply a `CASE WHEN` to assign a verification status label. The logic differs by warning type:

**TOR warnings:**
- Tornado reports present → Verified
- No tornado reports, but hail reports present → Unverified - Tornado w/ Hail
- No tornado reports, but wind reports present → Unverified - Tornado w/ Wind
- Nothing → Unverified - No Reports

**SVR warnings:**
- Hail OR wind reports present → Verified
- Nothing → Unverified - No Reports

Also add a boolean `is_verified` column — you'll use this for the WFO stats calculation.

Build this as another CTE wrapping your aggregation from Step 7.

In [10]:
class_v_warns = con.sql('''
    SELECT wfo, etn, warn_type,
        CASE WHEN warn_type = 'TO' AND tornado_reports > 0 THEN 'Verified'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND wind_reports > 0 AND hail_reports > 0) THEN 'Unverified - Tornado Warning w/ Hail + Wind'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND hail_reports > 0) THEN 'Unverified - Tornado Warning w/ Hail'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND wind_reports > 0) THEN 'Unverified - Tornado Warning w/ Wind'
             WHEN warn_type = 'TO' AND (tornado_reports = 0 AND wind_reports = 0 AND hail_reports = 0) THEN 'Unverified - No Reports'           
             WHEN warn_type = 'SV' AND (wind_reports > 0 OR hail_reports > 0) THEN 'Verified'
             ELSE 'Unverified - No Reports'
             END AS verification_status,
        CASE WHEN verification_status LIKE 'Verified%' THEN True
             ELSE False
             END AS is_verified,
        tornado_reports, hail_reports, wind_reports, total_reports, warn_geom
    FROM warn_lsr_totals                       
''')

class_v_warns.show()
con.sql('''SELECT DISTINCT verification_status, COUNT(*), is_verified
        FROM class_v_warns
        GROUP BY verification_status, is_verified''')

class_v_warns.show()

┌─────────┬───────┬───────────┬─────────────────────────┬─────────────┬─────────────────┬──────────────┬──────────────┬───────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│   WFO   │  ETN  │ WARN_TYPE │   verification_status   │ is_verified │ tornado_reports │ hail_reports │ wind_reports │ total_reports │                                                                                                                                                  warn_geom                                                                                                                                                   │
│ varchar │ int32 │  varchar  │         varchar         │   boolean   │     int128      │    int128    │    int128    │   

## Step 9: Per-WFO Verification Stats

Roll up the classified warnings to the WFO level. Calculate total warnings issued, verified warnings, and verification rate per WFO and warning type.

Keep the raw counts alongside the rate — a WFO that issued 3 warnings tells a very different story than one that issued 40, even if their rates look similar.

In [11]:
# WFO verification stats
wfo_stats = con.sql('''SELECT wfo, warn_type, count(*) as total_warnings, SUM(is_verified::INT) AS verified_warnings, ROUND(AVG(is_verified::INT) * 100,2) AS verification_rate_pct
                    FROM class_v_warns
                    GROUP BY wfo, warn_type
                    ORDER BY warn_type, verification_rate_pct DESC''')
wfo_stats.show(max_rows=41)

┌─────────┬───────────┬────────────────┬───────────────────┬───────────────────────┐
│   WFO   │ WARN_TYPE │ total_warnings │ verified_warnings │ verification_rate_pct │
│ varchar │  varchar  │     int64      │      int128       │        double         │
├─────────┼───────────┼────────────────┼───────────────────┼───────────────────────┤
│ HUN     │ SV        │              1 │                 1 │                 100.0 │
│ SHV     │ SV        │              2 │                 2 │                 100.0 │
│ MRX     │ SV        │              1 │                 1 │                 100.0 │
│ RLX     │ SV        │              1 │                 1 │                 100.0 │
│ LOT     │ SV        │              2 │                 2 │                 100.0 │
│ ARX     │ SV        │              1 │                 1 │                 100.0 │
│ LMK     │ SV        │             21 │                18 │                 85.71 │
│ IND     │ SV        │             36 │                30 │     

## Step 10: Export Final GeoJSON

Pull the classified warnings into a GeoPandas GeoDataFrame and write to GeoJSON. Each feature should carry the verification status, is_verified flag, report counts, WFO, warning type, and timing columns.

You'll need to reconstruct the geometry from the WKB column that came through the DuckDB pipeline before GeoPandas can write it out.

In [12]:
# Final warning table
final_warn = con.sql('''SELECT a.wfo, a.warn_type, a.verification_status, a.is_verified, a.tornado_reports, a.hail_reports, a.wind_reports, a.total_reports, b.issued, b.expired, a.warn_geom
                    FROM class_v_warns a
                    INNER JOIN ddb_warn b
                        ON a.wfo = b.wfo
                        AND a.etn = b.etn
                    GROUP BY a.wfo, a.warn_type, b.issued, b.expired, a.warn_geom, a.is_verified, a.verification_status, a.tornado_reports, a.hail_reports, a.wind_reports, a.total_reports
                    ''')

# Ensure geom column is converted to WKB blob prior to df
final_warn_df = con.execute("SELECT * EXCLUDE warn_geom, ST_asWKB(warn_geom) AS warn_geom FROM final_warn").df()

# Convert 'warn_geom' to Shapely geometries
final_warn_df['geometry'] = final_warn_df['warn_geom'].apply(lambda x: wkb.loads(bytes(x)))

# Create new GDF
final_warn_gdf = gpd.GeoDataFrame(final_warn_df, geometry='geometry', crs='EPSG:4326')

# Dropping original 'warn_geom' column
final_warn_gdf = final_warn_gdf.drop(columns=['warn_geom'])

# Export to GeoJSON
geojson_path = os.path.join("data", "processed", "verified_warnings.geojson")
final_warn_gdf.to_file(geojson_path, driver='GeoJSON')

# Export WFO stats to a CSV


## Step 11: Interactive Map with Folium

Build an interactive map using a traffic-light color scheme on the warning polygons:

- 🟢 **Green** — Verified
- 🔴 **Red** — Unverified, No Reports
- 🟡 **Yellow/Orange** — Unverified with partial reports (hail or wind inside a tornado warning)

Each polygon should have a popup or tooltip showing the WFO, warning type, report counts, and status label.

Center the map on the Midwest since that's where the bulk of April 27 activity occurred. Save the output as an HTML file.

---

## Stretch Goals

If you want to extend the analysis, here are two natural next steps:

### Lead Time Calculation
For each verified warning, calculate how much advance notice the public had — the time between `issue_time` and the first qualifying `report_time` inside the polygon.

### Warning-to-Report Distance
For unverified warnings, how close was the nearest qualifying report? Use `ST_Distance` to find the distance from each unverified polygon to its nearest LSR point, even if it fell outside the polygon boundary. This can surface "near miss" warnings where the storm tracked just outside the drawn polygon.

### Tornado Reports with no warning issued
